In [1]:
import numpy as np
from itertools import permutations        #generates every possible ordering for a collection of objects


#FIRST COMPOSING SIMPLE N=1,2,3 CASES TO FAMILIARISE MYSELF WITH THE CODE REQUIRED FOR THE ASSIGNMENT.


#Calculating the determinant of a completed matrix/finnished game for Matrix A, represented as board:
def determinant(A):

    #Gets the nunmber of rows of our matrix (as we're dealing with square matrix this gets us n)
    n = len(A)

    #Base case n=1: a 1x1 matrix
    if n == 1:
        return A[0][0]   #return the first row and first collumn value: the only value in the matrix

    total = 0

    #Looking across j collumnal values in first row
    for j in range(n):

        #construct smaller matrix, minor, by removing row 0 and column j, to be used in finding the determinant:
        #For an nxn matrix, expanding along the first row gives: det(A) = sum_{j=0}^{n} (-1)^{j} a_{0j} det(Minor)
        minor = []

        #for all rows excluding the first one, as we've excluded it.
        for i in range(1, n):
            #create store for row
            row = []

            #iterate thorugh the current row's collumn
            for k in range(n):

                 #only include entries that are not in the column currently being removed.
                if k != j:
                    row.append(A[i][k]) #adds each retained entry to the smaller matrix, 
                    #and then adds the completed row to the minor.
            minor.append(row)

        # Finding the sum: det(A) = sum_{j=0}^{n} (-1)^{j} a_{0j} det(Minor)
        total += (-1)**j * A[0][j] * determinant(minor)

    return total





# Checking whether the board is full to know whether to find determinant to determine winner or whether to go through with the next move.

def board_full(board):

    #for each cell in each row
    for row in board:
        for cell in row:

            #If there's no value in a cell, the matrix isn't full yet
            if cell is None:
                return False

    #otherwise it is full :D
    return True





#Determining whether I, Player A, can force a win


def can_win(board, my_turn):

    #counting rows in Matrix Board.
    n = len(board)

    #If the board is full, check the determinant, as the game is over and we need to determine the winner.
    if board_full(board):
        return determinant(board) == 0

    #Now, otherwise, this is a playing move: consider every cell.
    for i in range(n):
        for j in range(n):

            #Only consider empty cells: if i-jth cell is empty:
            if board[i][j] is None:

                #Make the move in this square: remember, we are running through every possible legal move for every possible i-j cell, this may not be a winning move/the optimal move to make but we investigate it as a LEGAL move
                if my_turn:
                    board[i][j] = 0 #Place a 0, As its my (PLayer A)'s turn
                else:
                    board[i][j] = 1  #PLace a 1, as its PLayer B's turn

                #Investigate the resulting position after this move:
                result = can_win(board, not my_turn) #not my_turn swaps true<->falsew: If i just moved, then Player B moves, if PLayer B moved, then I move.

                #Undo the move: after the move and its result has been considered ^ we undo the placement so we can check what'll happen if I place it in any of the other empty cells without this^ investigative placement remaining there.
                board[i][j] = None

                #My turn: one successful move is enough
                if my_turn and result:
                    return True #There's a winning move! We only need to find one placement that is winning: even if there's more we've done what we need already :)

                #Player B's turn: ther're playing optimally so if there's even one guaranteed failure route available from this position then PLayer B will win: therefore this position is not a winning position.
                if not my_turn and not result:
                    return False
                    #We've now thoroughly checked position i-j for whether its both our turn or if its player B's turn.

    #If its my move and we've checked every possible placement and none come back true (no winning move available) then we know this, current, position is not a winning one, and so we return false
    if my_turn:
        return False

    #Otherwise there is a guaranteed route to success: regardless of what Player B does we're guaranteed a win. This is a winning position. Return True.
    return True







# Finding an actual winning move: we've figured out how to check whether a position at any point in the game is a winning one, now we need to find it

def find_winning_move(board):

    n = len(board)

    #for cell in i row j collmn
    for i in range(n):
        for j in range(n):

            #If cell i-j is empy
            if board[i][j] is None:

                #Temporarily place a zero, it is our turn, we're again checking viability of the position resulting in this move.
                board[i][j] = 0

                #Check whether this move guarantees a win
                result = can_win(board, False)

                #Undo the move again
                board[i][j] = None

                #If we get a winning position out of this move, return the i-j position we want to place our 0 in: that's the move we want to make
                if result:
                    return (i, j)

    #No winning move exists
    return None




#The previous function spits out the first winning move it comes across, not ALL winning moves: we want to find ALL winning first moves

def find_all_winning_moves(board):

    n = len(board)

    #creating a store for all possible winning moves
    winning_moves = []

    for i in range(n):
        for j in range(n):

            #checking each empty cell
            if board[i][j] is None:

                #Temporarily place a zero
                board[i][j] = 0

                #Check whether this move guarantees a win
                result = can_win(board, False)

                #Undo the move
                board[i][j] = None

                #add to store
                if result:
                    winning_moves.append((i, j))

    #return all winning moves found
    return winning_moves


  
# Testing different grid sizes n=1,2,3

for n in range(1, 4):

    #empty store to put matrix in
    board = []

    #for row i
    for i in range(n):

        #empty store
        row = []

        #j'th collumnal position in row i
        for j in range(n):
            row.append(None) 

        board.append(row)
        #inserted matrix into board

    result = can_win(board, True)

  

    print("n=", n)

    if result:
        print("PLayer A can guarantee a win")

        winning_moves = find_all_winning_moves(board)

        print("All winning first moves:", winning_moves)

    else:
        print("There is no guaranteed win for PLayer A-- Player B may win")

    print()




n= 1
PLayer A can guarantee a win
All winning first moves: [(0, 0)]

n= 2
PLayer A can guarantee a win
All winning first moves: [(0, 0), (0, 1), (1, 0), (1, 1)]

n= 3
PLayer A can guarantee a win
All winning first moves: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]



In [2]:
#We've identifyed for n=1,2,3 that Player A can place their first 0 anywhere and they'll still win when playing optimally
#As a little investigation we'll simulate Player A placing their first 0 in matrix position (0,0) with matrix n=3, seeing any possible Player B responses that could be made and what I should do in response


#Investigating opponent responses to n=3 matrix
f = 3

#Empty store to put in matrix
board = []

#creating the matrix
for i in range(f):
    row = []
    for j in range(f):
        row.append(None)
    board.append(row)

#My first move, as advised, is placing a 0 in (0,0)
board[0][0] = 0


for i in range(f):
    for j in range(f):

        #investigating empty cell i-j
        if board[i][j] is None:

            #Investigate what happens here when PLayer B places a 1
            board[i][j] = 1

            #Finding Player A's next winning move given Player B's response.
            move = find_winning_move(board)

            print("Opponent:", (i, j))
            print("My response:", move)

            #Undo opponent's move so we can go on and investigate all other possible PLayer B responses
            board[i][j] = None



Opponent: (0, 1)
My response: (0, 2)
Opponent: (0, 2)
My response: (0, 1)
Opponent: (1, 0)
My response: (0, 1)
Opponent: (1, 1)
My response: (0, 1)
Opponent: (1, 2)
My response: (0, 1)
Opponent: (2, 0)
My response: (0, 1)
Opponent: (2, 1)
My response: (0, 1)
Opponent: (2, 2)
My response: (0, 1)


In [3]:
#Whilst this is initially useful, it only provides us up to our second possible move given one specific starting move.
#Not only does it not provide a complete strategy for winning as Player A on a 3x3 Matrix, but we're no where near being able to expand generalised strategies to greater matricies.

#We've already shown that first zero does not matter for n=3, however, I don't want to jump to the conclusion that all subsequent moves are equally viable



#I will instead be investigating an attempt at generating a complete winning strategy
#need indent to create a somewhat legible tree of reasoning

def print_strategy(board, my_turn, indent=""):

    n = len(board)

    #Check whether the game has already ended, and whether its time to determine the winner.
    if board_full(board):
        if determinant(board) == 0:
            print(indent + "Game over. Player A wins:")

        else: print(indent + "Game over. Player B wins")
    
        return

    #PLAYER A'S TURN:
    if my_turn:

        #Find one move that guarantees a win: this already accounts for checking for empty cells
        move = find_winning_move(board)

        #if there does not exist such a move:
        if move is None:
            print(indent + "No guaranteed winning move exists.")
            return

        #otherwise, we make an indent for an 'in this condition' tree flow and showcase what move should be made next.
        i, j = move
        print(indent + "Player A: place 0 at", move)

        #Make the move
        board[i][j] = 0

        #Continue with Player B's turn
        print_strategy(board, False, indent)

        #Undo the move after investigating this branch
        board[i][j] = None

    #PLAYER B'S TURN
    else:

        #Investigate every possible response from Player B
        for i in range(n):
            for j in range(n):

                #if the position is empty
                if board[i][j] is None:

                    #Player B places a 1 - we're accounting for every possible move Player B could make.
                    board[i][j] = 1

                    #'in this condition' tree flor
                    print(indent + "If Player B places 1 at", (i, j), ":")

                    #continue with Player A's turn
                    print_strategy(board, True, indent + "    ")

                    # Undo Player B's move before checking another response
                    board[i][j] = None

#as, as we know, any starting position on a 3x3 is a winning position for Player A, this provides us a guide on how to win EVERY TIME. We should not get the message at the end of a tree 'Game over. Player B wins'

#Creating an empty matrix of any size
def create_board(n):

    board = []

    for i in range(n):

        row = []

        for j in range(n):
            row.append(None)

        board.append(row)

    return board



#Generating the complete strategy for n=3

n = 3

board = create_board(n)

print("COMPLETE WINNING STRATEGY FOR", n, "x", n)
print()

if can_win(board, True):
    print_strategy(board, True)

else:
    print("Player A cannot guarantee a win.")


COMPLETE WINNING STRATEGY FOR 3 x 3

Player A: place 0 at (0, 0)
If Player B places 1 at (0, 1) :
    Player A: place 0 at (0, 2)
    If Player B places 1 at (1, 0) :
        Player A: place 0 at (2, 2)
        If Player B places 1 at (1, 1) :
            Player A: place 0 at (1, 2)
            If Player B places 1 at (2, 0) :
                Player A: place 0 at (2, 1)
                Game over. Player A wins:
            If Player B places 1 at (2, 1) :
                Player A: place 0 at (2, 0)
                Game over. Player A wins:
        If Player B places 1 at (1, 2) :
            Player A: place 0 at (2, 0)
            If Player B places 1 at (1, 1) :
                Player A: place 0 at (2, 1)
                Game over. Player A wins:
            If Player B places 1 at (2, 1) :
                Player A: place 0 at (1, 1)
                Game over. Player A wins:
        If Player B places 1 at (2, 0) :
            Player A: place 0 at (1, 2)
            If Player B places

In [ ]:
# Investigating whether Player A can force a win for different values of n

for n in range(1, 6):

    board = create_board(n)

    # By symmetry, we only need to investigate one possible first move.
    # I will start by placing a zero in position (0,0).
    board[0][0] = 0

    # It is now Player B's turn.
    result = can_win(board, False)

    print("n =", n)

    if result:
        print("Player A can guarantee a win.")
    else:
        print("Player A cannot guarantee a win.")

    print()

n = 1
Player A can guarantee a win.

n = 2
Player A can guarantee a win.

n = 3
Player A can guarantee a win.

